# Music Recommendation — Content-based Only

This notebook builds a **content-based** music recommender using TF-IDF on combined text (song, artist, genre, lyrics) and cosine similarity.

In [1]:
import os 
print(os.path.abspath("S24CSEU2599          MEHUL JAKHMOLA CODE.ipynb"))

C:\Users\M!KK_J\AI & Society Project\S24CSEU2599          MEHUL JAKHMOLA CODE.ipynb


In [2]:

# Imports
import numpy as np
import pandas as pd
import joblib, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

C:\Users\M!KK_J\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


KeyboardInterrupt: 

In [ ]:
# Load dataset (mapping 'text' column to lyrics)
DATA_PATH = "spotify_millsongdata_with_genres.csv"
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower() for c in df.columns]
# map 'text' -> 'lyrics' if present
if 'text' in df.columns and 'lyrics' not in df.columns:
    df = df.rename(columns={'text':'lyrics'})
expected = {'artist','song','genre','lyrics'}
missing = expected - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}. Required: {expected}")
print("Loaded rows:", len(df))
df.head(5)

In [ ]:

# Basic preprocessing and combined text
df['artist'] = df['artist'].astype(str).fillna('').str.strip()
df['song']   = df['song'].astype(str).fillna('').str.strip()
df['genre']  = df['genre'].astype(str).fillna('').str.strip()
df['lyrics'] = df['lyrics'].astype(str).fillna('').str.strip()
df['text'] = (df['song'] + ' ' + df['artist'] + ' ' + df['genre'] + ' ' + df['lyrics']).str.lower()
df = df.reset_index(drop=True)
df['song_id'] = df.index.map(lambda x: f"S{str(x).zfill(5)}")
print("Prepared text — example:")
print(df['text'].iloc[0][:300])

In [ ]:

# TF-IDF vectorization (fit)
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_text = tfidf.fit_transform(df['text'])
print("TF-IDF matrix shape:", X_text.shape)
joblib.dump(tfidf, 'tfidf_vectorizer_contentonly.pkl')
print("Saved tfidf_vectorizer_contentonly.pkl")

In [ ]:

# Content-based recommender functions
from sklearn.metrics.pairwise import linear_kernel
songid_to_idx = {sid: idx for idx, sid in enumerate(df['song_id'])}

def recommend_by_song(song_id, top_n=10):
    if song_id not in songid_to_idx:
        raise ValueError("song_id not found")
    idx = songid_to_idx[song_id]
    sims = cosine_similarity(X_text[idx], X_text).flatten()
    sims[idx] = -1
    top_idx = np.argpartition(-sims, range(min(top_n, len(sims))))[:min(top_n, len(sims))]
    top_sorted = top_idx[np.argsort(-sims[top_idx])]
    return df.iloc[top_sorted].copy().assign(score = sims[top_sorted])

def recommend_by_text(seed_text, top_n=10):
    vec = tfidf.transform([seed_text.lower()])
    sims = cosine_similarity(vec, X_text).flatten()
    top_idx = np.argpartition(-sims, range(min(top_n, len(sims))))[:min(top_n, len(sims))]
    top_sorted = top_idx[np.argsort(-sims[top_idx])]
    return df.iloc[top_sorted].copy().assign(score=sims[top_sorted])

example_song = df.loc[0,'song_id']
print('Example seed song:', example_song, df.loc[0,['song','artist','genre']].to_dict())
recommend_by_song(example_song, top_n=5)

In [ ]:

# Example: recommend by custom text (e.g., "upbeat pop dance")
seed = "upbeat pop dance"
print("Seed text:", seed)
recommend_by_text(seed, top_n=5)

In [ ]:

# Save top-5 recommendations for the first 10 songs to a CSV (example output)
outputs = []
for i in range(min(10, len(df))):
    sid = df.loc[i,'song_id']
    recs = recommend_by_song(sid, top_n=5)
    for rank, (_, r) in enumerate(recs.iterrows(), start=1):
        outputs.append({'seed_song_id': sid, 'seed_title': df.loc[i,'song'], 'rank': rank, 'rec_song_id': r['song_id'], 'rec_title': r['song'], 'rec_artist': r['artist'], 'rec_genre': r['genre'], 'score': r['score']})
outdf = pd.DataFrame(outputs)
outdf.to_csv('example_recommendations_top5_for_first10.csv', index=False)
print('Wrote example_recommendations_top5_for_first10.csv')

In [ ]:

# Quick utility: show recommendations neatly for a given song index (human-friendly)
def pretty_show(song_idx, top_n=5):
    sid = df.loc[song_idx, 'song_id']
    print("Seed:", sid, "-", df.loc[song_idx,['song','artist','genre']].to_dict())
    print(recommend_by_song(sid, top_n=top_n)[['song_id','song','artist','genre','score']].to_string(index=False))

for i in range(min(3, len(df))):
    pretty_show(i, top_n=5)
    print('---')

song_id                song         artist   genre    score
 S48822         Sing A Song Point Of Grace Unknown 0.494453
 S26224 Let It All Be Music       Boney M. Unknown 0.473766
 S04271           Applejack   Dolly Parton Unknown 0.410842
 S45324         I Am A Song    Neil Sedaka Unknown 0.405204
 S26919                Sing     Carpenters Unknown 0.400618
---
Seed: S00002 - {'song': 'As Good As New', 'artist': 'ABBA', 'genre': 'Pop'}
song_id               song                   artist   genre    score
 S01819           Ma Baker                 Boney M. Unknown 0.904975
 S54116 Big Girls Are Best                       U2 Unknown 0.895556
 S32935       Big Leg Emma              Frank Zappa Unknown 0.862077
 S34945   Move To The City            Guns N' Roses Unknown 0.822153
 S04727           Auntie - Electric Light Orchestra Unknown 0.808869
---


# Music Recommendation UI........................

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# MAIN UI ELEMENTS
input_type = widgets.Dropdown(
    options=[
        "Search by Song Name",
        "Search by Song ID",
        "Search by Artist Name",   # NEW MODE
        "Custom Text Query"
    ],
    value="Search by Song Name",
    description="Mode:"
)

user_input = widgets.Text(
    placeholder="Enter song name, artist name, ID, or description...",
    description="Input:",
    layout=widgets.Layout(width='60%')
)

num_results = widgets.IntSlider(
    value=10, min=3, max=20, step=1,
    description="Top N:"
)

recommend_button = widgets.Button(
    description="Recommend",
    button_style="success",
    tooltip="Click to get recommendations"
)

random_button = widgets.Button(
    description="🎲 Random Song Recommend",
    button_style="info",
    tooltip="Pick a random song and recommend"
)

output = widgets.Output()

# -----------------------------------
# HANDLERS
# -----------------------------------

def recommend_from_ui(_):
    with output:
        clear_output()
        mode = input_type.value
        text = user_input.value.strip()
        n = num_results.value
        
        if text == "":
            print("⚠️ Please enter something.")
            return
        
        # MODE 1: search by song name
        if mode == "Search by Song Name":
            matches = df[df['song'].str.contains(text, case=False, na=False)]
            if len(matches) == 0:
                print("❌ No song found with that name.")
                return
            sid = matches.iloc[0]['song_id']
            print(f"🎵 Matched Song: {sid} – {matches.iloc[0]['song']}")
            display(recommend_by_song(sid, top_n=n)[['song_id','song','artist','genre','score']])
        
        # MODE 2: search by song ID
        elif mode == "Search by Song ID":
            if text not in df['song_id'].values:
                print("❌ Invalid song ID.")
                return
            print(f"🎵 Seed Song: {text} – {df.loc[df['song_id']==text, 'song'].values[0]}")
            display(recommend_by_song(text, top_n=n)[['song_id','song','artist','genre','score']])

        # MODE 3: search by ARTIST NAME
        elif mode == "Search by Artist Name":
            matches = df[df['artist'].str.contains(text, case=False, na=False)]
            if len(matches) == 0:
                print("❌ No songs found for that artist.")
                return

            seed_row = matches.iloc[0]
            sid = seed_row['song_id']
            print(f"🎤 Artist Match: {seed_row['artist']}")
            print(f"🎵 Using song: {sid} – {seed_row['song']}")
            display(recommend_by_song(sid, top_n=n)[['song_id','song','artist','genre','score']])

        # MODE 4: custom text query
        elif mode == "Custom Text Query":
            print(f"📝 Query: {text}")
            display(recommend_by_text(text, top_n=n)[['song_id','song','artist','genre','score']])


def recommend_random(_):
    with output:
        clear_output()
        # Pick random song
        idx = np.random.randint(0, len(df))
        sid = df.loc[idx, 'song_id']
        title = df.loc[idx, 'song']
        print(f"🎲 Random Seed: {sid} – {title}")
        display(recommend_by_song(sid, top_n=num_results.value)[['song_id','song','artist','genre','score']])


# CONNECT BUTTONS
recommend_button.on_click(recommend_from_ui)
random_button.on_click(recommend_random)

# DISPLAY UI
display(input_type, user_input, num_results, recommend_button, random_button, output)


In [ ]:
df[df['genre'] != 'Unknown']['genre'].value_counts().head(10).plot(kind='bar', figsize=(12,6))
plt.title("Top 10 Genres (Excluding 'Unknown')")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()



In [ ]:
df[df['genre'] != 'Unknown']['genre'].value_counts().head(8).plot(
    kind='pie', autopct='%1.1f%%', figsize=(8,8)
)
plt.title("Genre Distribution (Without Unknown)")
plt.ylabel("")
plt.show()


In [ ]:
from collections import Counter
import re

all_words = " ".join(df['lyrics']).lower()
all_words = re.findall(r"[a-z']+", all_words)
common = Counter(all_words).most_common(20)

words = [w for w,c in common]
counts = [c for w,c in common]

plt.figure(figsize=(10,5))
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top 20 Most Frequent Words in Lyrics")
plt.show()


In [ ]:
feature_names = tfidf.get_feature_names_out()
row = X_text[0].toarray().ravel()
top_idx = row.argsort()[-20:]
plt.barh(feature_names[top_idx], row[top_idx])
plt.title("Top TF-IDF Words for Song 0")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# choose N (20 was used before); try 30 or 40 for clearer patterns
N = 30
Xsmall = X_text[:N]
sim = cosine_similarity(Xsmall)  # NxN matrix

plt.figure(figsize=(10,8))
sns.set(style="white")
ax = sns.heatmap(sim,
                 cmap="coolwarm",      # pleasant continuous colormap
                 vmin=0, vmax=1,      # similarity in [0,1]
                 square=True,
                 linewidths=0.4,      # subtle grid lines
                 linecolor='white',
                 cbar_kws={'shrink':0.6, 'label':'cosine similarity'})

ax.set_title(f"Cosine Similarity Heatmap (First {N} Songs)", fontsize=14)
ax.set_xlabel("Song index")
ax.set_ylabel("Song index")
plt.tight_layout()
plt.savefig("heatmap_clean.png", dpi=300)
plt.show()
